In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Indian_startups_funding.csv")

df.head()

,Unnamed: 0,Sr No,startup,vertical,city,investors,round,amount,year,state
0,0,1,#Fame,Media,Mumbai,To the new ventures,Private Equity,82.50,2015,Maharashtra
1,1,2,1mg,Healthcare,Gurugram,"Deep kalra, Sequoia capital, Omidyar network, ...",Private Equity,49.50,2015,Haryana
2,2,3,20Dresses,Personal Care,Mumbai,"Srinivas aluri, Srinivas modukury, Bs nagesh",Seed Funding,8.25,2015,Maharashtra
3,3,4,33Coupons,Marketing,Kanpur,Not disclosed,Seed Funding,0.17,2015,Uttar Pradesh
4,4,5,6Degree,Networking,Mumbai,Centre for incubation & business acceleration ...,Seed Funding,0.00,2015,Maharashtra


In [2]:
df = df.drop(columns=["Unnamed: 0", "Sr No"])

df.shape

(5074, 8)

In [3]:
text_columns = [
    "startup",
    "vertical",
    "city",
    "investors",
    "round",
    "state"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

In [4]:
df["vertical"] = df["vertical"].replace({
    "Fintech": "FinTech"
})

In [5]:
df["round"] = df["round"].replace({
    "Seed funding": "Seed Funding",
    "Crowd funding": "Crowd Funding",
    "pre-series A": "Pre-Series A",
    "pre-Series A": "Pre-Series A",
    "Pre-series A": "Pre-Series A"
})

In [6]:
df["investors"] = df["investors"].fillna("Undisclosed")

In [7]:
df["disclosed_amount"] = df["amount"].replace(0, np.nan)

In [8]:
df["funding_disclosed"] = np.where(
    df["amount"] > 0,
    "Disclosed",
    "Undisclosed"
)

In [9]:
df["funding_disclosed"].value_counts()

funding_disclosed
Disclosed      3475
Undisclosed    1599
Name: count, dtype: int64

In [10]:
def funding_bucket(amount):
    if pd.isna(amount) or amount == 0:
        return "Undisclosed"
    elif amount < 10:
        return "< ₹10 Cr"
    elif amount < 50:
        return "₹10–50 Cr"
    elif amount < 100:
        return "₹50–100 Cr"
    elif amount < 500:
        return "₹100–500 Cr"
    else:
        return "₹500+ Cr"

df["funding_size"] = df["amount"].apply(funding_bucket)

In [11]:
df["startup"] = df["startup"].str.title()

In [12]:
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nFunding Disclosure:")
print(df["funding_disclosed"].value_counts())

print("\nTop Sectors:")
print(df["vertical"].value_counts().head(15))

print("\nFunding Rounds:")
print(df["round"].value_counts().head(20))

(5074, 11)

Missing Values:
startup                 0
vertical                0
city                    0
investors               0
round                   0
amount                  0
year                    0
state                   0
disclosed_amount     1599
funding_disclosed       0
funding_size            0
dtype: int64

Funding Disclosure:
funding_disclosed
Disclosed      3475
Undisclosed    1599
Name: count, dtype: int64

Top Sectors:
vertical
Consumer Internet    738
Technology           578
E-Commerce           522
Healthcare           397
FinTech              343
Finance              250
Food                 243
EdTech               234
Logistics            136
Transportation       120
Personal Care        104
AI                    95
Services              89
AgriTech              83
Entertainment         82
Name: count, dtype: Int64

Funding Rounds:
round
Private Equity              1354
Seed Funding                1348
Undisclosed                 1082
Seed                  

In [13]:
sector_mapping = {
    # FinTech & Finance
    "FinTech": "FinTech",
    "Finance": "FinTech",
    "Banking": "FinTech",

    # Education
    "EdTech": "EdTech",
    "Education": "EdTech",
    "Careers": "EdTech",

    # Healthcare
    "Healthcare": "HealthTech",
    "BioTech": "HealthTech",

    # E-Commerce / Retail
    "E-Commerce": "E-Commerce & Retail",
    "Retail": "E-Commerce & Retail",

    # Enterprise / Software / IT
    "SaaS": "Enterprise Tech",
    "Software": "Enterprise Tech",
    "IT": "Enterprise Tech",
    "Cloud Tech": "Enterprise Tech",
    "Cybersecurity": "Enterprise Tech",
    "B2B": "Enterprise Tech",
    "HR Tech": "Enterprise Tech",
    "Networking": "Enterprise Tech",

    # AI / Data / Advanced Tech
    "AI": "AI & Data",
    "Data Analytics": "AI & Data",
    "IoT": "AI & Data",
    "Robotics": "AI & Data",

    # Food
    "Food": "FoodTech",

    # Logistics / Delivery
    "Logistics": "Logistics & Delivery",
    "Delivery": "Logistics & Delivery",
    "Log Management": "Logistics & Delivery",

    # Mobility
    "Transportation": "Mobility",
    "Automotive": "Mobility",
    "EV": "Mobility",
    "Airlines": "Mobility",

    # Agriculture
    "AgriTech": "AgriTech",
    "Agriculture": "AgriTech",

    # Consumer
    "Consumer Internet": "Consumer Internet",
    "Consumer Goods": "Consumer & Lifestyle",
    "Personal Care": "Consumer & Lifestyle",
    "Lifestyle": "Consumer & Lifestyle",
    "Home & Lifestyle": "Consumer & Lifestyle",
    "Households": "Consumer & Lifestyle",
    "Pet Care": "Consumer & Lifestyle",

    # Media / Entertainment
    "Entertainment": "Media & Entertainment",
    "Media": "Media & Entertainment",
    "Content Creation": "Media & Entertainment",
    "Social Media": "Media & Entertainment",
    "Sports": "Media & Entertainment",
    "Dating": "Media & Entertainment",

    # Travel
    "Travel": "Travel & Hospitality",
    "Hospitality": "Travel & Hospitality",

    # Real Estate / Construction
    "Real Estate": "Real Estate & Construction",
    "Construction": "Real Estate & Construction",
    "Construction Tech": "Real Estate & Construction",

    # Marketing
    "Marketing": "Marketing & Advertising",
    "Design": "Marketing & Advertising",
    "Event Management": "Marketing & Advertising",

    # Energy / Sustainability
    "Energy": "Energy & Sustainability",
    "Waste Management": "Energy & Sustainability",
    "Water Solutions": "Energy & Sustainability",

    # Industrial / Deep Tech
    "Manufacturing": "Industrial & DeepTech",
    "Electronics": "Industrial & DeepTech",
    "Aerospace": "Industrial & DeepTech",

    # Other technology
    "Technology": "Technology",
    "Internet": "Technology",
    "Communication": "Technology",

    # Services
    "Services": "Services",
    "Consulting": "Services",
    "Gig Economy": "Services",

    # Other
    "Crowdfunding": "Other",
    "Charity": "Other"
}

df["sector_group"] = df["vertical"].map(sector_mapping)

df["sector_group"] = df["sector_group"].fillna("Other")

In [14]:
df["sector_group"].value_counts()

sector_group
Consumer Internet             738
Technology                    611
FinTech                       598
E-Commerce & Retail           524
HealthTech                    408
EdTech                        341
FoodTech                      243
Media & Entertainment         233
Mobility                      197
Enterprise Tech               195
Logistics & Delivery          191
AI & Data                     160
Consumer & Lifestyle          135
Services                       96
Marketing & Advertising        94
Travel & Hospitality           94
AgriTech                       93
Real Estate & Construction     54
Energy & Sustainability        33
Industrial & DeepTech          32
Other                           4
Name: count, dtype: int64

In [15]:
def classify_round(round_name):

    if pd.isna(round_name):
        return "Undisclosed"

    round_name = str(round_name).strip()

    # Pre-seed / seed / angel
    if round_name in [
        "Pre-Seed",
        "Seed",
        "Seed Funding",
        "Seed Round",
        "Angel",
        "Angel Funding",
        "Angel Round",
        "Angel / Seed Funding",
        "Seed / Angel Funding",
        "Seed / Angle Funding",
        "Seed/ Angel Funding",
        "Seed/Angel Funding",
        "Maiden Round"
    ]:
        return "Seed / Angel"

    # Early stage
    elif round_name in [
        "Pre-Series A",
        "Pre Series A",
        "Series A",
        "Bridge Round"
    ]:
        return "Early Stage"

    # Growth
    elif round_name in [
        "Series B",
        "Series B (Extension)",
        "Series C"
    ]:
        return "Growth Stage"

    # Late stage
    elif round_name in [
        "Series D",
        "Series E",
        "Series F",
        "Series G",
        "Series H",
        "Series J"
    ]:
        return "Late Stage"

    # Debt
    elif round_name in [
        "Debt",
        "Debt Financing",
        "Debt Funding",
        "Debt-Funding",
        "Structured Debt",
        "Term Loan",
        "Debt and Preference capital",
        "Mezzanine",
        "Post-IPO Debt"
    ]:
        return "Debt"

    # Private equity
    elif round_name in [
        "Private Equity",
        "PrivateEquity",
        "Private",
        "Private Funding"
    ]:
        return "Private Equity"

    # Venture / unknown series
    elif round_name in [
        "Venture",
        "Venture Round",
        "Venture - Series Unknown",
        "Single Venture"
    ]:
        return "Venture - Unspecified"

    # Public / secondary
    elif round_name in [
        "Post-IPO Equity",
        "Secondary Market"
    ]:
        return "Public / Secondary"

    # Grants / non-equity
    elif round_name in [
        "Grant",
        "Non-equity Assistance",
        "Inhouse Funding"
    ]:
        return "Grant / Non-Equity"

    # Corporate
    elif round_name == "Corporate Round":
        return "Corporate"

    # Crowd
    elif round_name == "Crowd Funding":
        return "Crowdfunding"

    # Unknown/general categories
    elif round_name in [
        "Undisclosed",
        "Funding Round",
        "Equity",
        "Equity Based Funding"
    ]:
        return "Undisclosed / Other"

    # Dataset corruption
    elif "http" in round_name.lower():
        return "Undisclosed / Other"

    else:
        return "Undisclosed / Other"


df["round_group"] = df["round"].apply(classify_round)

In [16]:
df["round_group"].value_counts()

round_group
Seed / Angel             2129
Private Equity           1357
Undisclosed / Other      1090
Early Stage               177
Growth Stage              104
Venture - Unspecified      79
Debt                       59
Late Stage                 36
Corporate                  18
Public / Secondary         12
Grant / Non-Equity         11
Crowdfunding                2
Name: count, dtype: int64

In [17]:
round_order = {
    "Seed / Angel": 1,
    "Early Stage": 2,
    "Growth Stage": 3,
    "Late Stage": 4,
    "Venture - Unspecified": 5,
    "Private Equity": 6,
    "Debt": 7,
    "Corporate": 8,
    "Public / Secondary": 9,
    "Grant / Non-Equity": 10,
    "Crowdfunding": 11,
    "Undisclosed / Other": 12
}

df["round_order"] = df["round_group"].map(round_order)

In [18]:
print("Sector Groups:")
print(df["sector_group"].value_counts())

print("\nFunding Round Groups:")
print(df["round_group"].value_counts())

print("\nMissing Sector Groups:")
print(df["sector_group"].isna().sum())

print("\nMissing Round Groups:")
print(df["round_group"].isna().sum())

Sector Groups:
sector_group
Consumer Internet             738
Technology                    611
FinTech                       598
E-Commerce & Retail           524
HealthTech                    408
EdTech                        341
FoodTech                      243
Media & Entertainment         233
Mobility                      197
Enterprise Tech               195
Logistics & Delivery          191
AI & Data                     160
Consumer & Lifestyle          135
Services                       96
Marketing & Advertising        94
Travel & Hospitality           94
AgriTech                       93
Real Estate & Construction     54
Energy & Sustainability        33
Industrial & DeepTech          32
Other                           4
Name: count, dtype: int64

Funding Round Groups:
round_group
Seed / Angel             2129
Private Equity           1357
Undisclosed / Other      1090
Early Stage               177
Growth Stage              104
Venture - Unspecified      79
Debt             

In [19]:
df["startup_clean"] = df["startup"]
df["city_clean"] = df["city"]
df["state_clean"] = df["state"]

In [20]:
city_mapping = {
    # Bengaluru
    "Bangalore": "Bengaluru",
    "Bangaluru": "Bengaluru",

    # Kochi
    "Cochin": "Kochi",
    "Ernakulam": "Kochi",

    # Thiruvananthapuram
    "Trivandrum": "Thiruvananthapuram",

    # Mumbai localities
    "Andheri": "Mumbai",
    "Chembur": "Mumbai",
    "Worli": "Mumbai",

    # Chennai locality
    "Taramani": "Chennai"
}

df["city_clean"] = df["city_clean"].replace(city_mapping)

In [21]:
df["startup_hub"] = df["city_clean"]

In [22]:
delhi_ncr_cities = [
    "New Delhi",
    "Gurugram",
    "Noida",
    "Ghaziabad",
    "Faridabad",
    "Azadpur"
]

df.loc[
    df["city_clean"].isin(delhi_ncr_cities),
    "startup_hub"
] = "Delhi NCR"

In [23]:
state_as_city = [
    "Haryana",
    "Kerala",
    "Uttar Pradesh"
]

df.loc[
    df["city_clean"].isin(state_as_city),
    "city_clean"
] = "City Not Specified"

In [24]:
df.loc[
    df["startup_hub"].isin(state_as_city),
    "startup_hub"
] = "Other / Unspecified"

In [25]:
df["state_clean"] = df["state_clean"].replace({
    "Pune": "Maharashtra"
})

In [26]:
df["startup_key"] = (
    df["startup"]
    .str.strip()
    .str.casefold()
)

In [27]:
canonical_names = (
    df.groupby(["startup_key", "startup"])
      .size()
      .reset_index(name="count")
      .sort_values(
          ["startup_key", "count"],
          ascending=[True, False]
      )
      .drop_duplicates("startup_key")
      .set_index("startup_key")["startup"]
)

In [28]:
df["startup_clean"] = df["startup_key"].map(canonical_names)

In [29]:
df[["startup", "startup_clean"]].head(20)

,startup,startup_clean
0,#Fame,#Fame
1,1Mg,1Mg
2,20Dresses,20Dresses
3,33Coupons,33Coupons
4,6Degree,6Degree
5,Aagaar,Aagaar
6,Aasaanjobs,Aasaanjobs
7,Abhibus,Abhibus
8,Absentia,Absentia
9,Actionable Analytics,Actionable Analytics


In [31]:
df = df.drop(columns=["startup_key"])

In [32]:
print("Top Startup Hubs:")
print(df["startup_hub"].value_counts().head(15))

Top Startup Hubs:
startup_hub
Bengaluru              1643
Delhi NCR              1502
Mumbai                  971
Chennai                 203
Pune                    194
Hyderabad               162
Ahmedabad                67
Jaipur                   63
Kolkata                  46
Indore                   28
Other / Unspecified      21
Kochi                    19
Chandigarh               17
Goa                      14
Vadodara                 13
Name: count, dtype: Int64


In [33]:
print("\nTop Clean Cities:")
print(df["city_clean"].value_counts().head(15))


Top Clean Cities:
city_clean
Bengaluru             1643
Mumbai                 971
New Delhi              691
Gurugram               636
Chennai                203
Pune                   194
Noida                  164
Hyderabad              162
Ahmedabad               67
Jaipur                  63
Kolkata                 46
Indore                  28
City Not Specified      21
Kochi                   19
Chandigarh              17
Name: count, dtype: Int64


In [34]:
print("\nStates:")
print(df["state_clean"].value_counts())


States:
state_clean
Karnataka            1650
Maharashtra          1181
Delhi                 698
Haryana               657
Tamil Nadu            216
Uttar Pradesh         184
Telangana             164
Gujarat                84
Rajasthan              77
West Bengal            48
Madhya Pradesh         38
Kerala                 31
Chandigarh             17
Goa                    16
Bihar                   3
Punjab                  3
Odisha                  3
Andhra Pradesh          2
Uttarakhand             1
Jammu and Kashmir       1
Name: count, dtype: Int64


In [36]:
print("Original unique startup names:")
print(df["startup"].nunique())

print("\nNormalized startup names:")
print(df["startup_clean"].nunique())

Original unique startup names:
3139

Normalized startup names:
3139


In [38]:
def funding_bucket(amount):
    if pd.isna(amount) or amount == 0:
        return "Undisclosed"
    elif amount < 10:
        return "< ₹10 Cr"
    elif amount < 50:
        return "₹10–50 Cr"
    elif amount < 100:
        return "₹50–100 Cr"
    elif amount < 500:
        return "₹100–500 Cr"
    else:
        return "₹500+ Cr"

df["funding_size"] = df["amount"].apply(funding_bucket)

In [39]:
funding_size_order = {
    "Undisclosed": 0,
    "< ₹10 Cr": 1,
    "₹10–50 Cr": 2,
    "₹50–100 Cr": 3,
    "₹100–500 Cr": 4,
    "₹500+ Cr": 5
}

df["funding_size_order"] = df["funding_size"].map(funding_size_order)

In [40]:
df["is_disclosed"] = (df["amount"] > 0).astype(int)

In [41]:
df["disclosed_amount"] = df["amount"].replace(0, np.nan)

In [42]:
df["mega_deal"] = np.where(
    df["amount"] >= 500,
    "Mega Deal",
    "Regular Deal"
)

In [43]:
df["mega_deal"] = np.where(
    df["amount"] == 0,
    "Undisclosed",
    np.where(
        df["amount"] >= 500,
        "Mega Deal",
        "Regular Deal"
    )
)

In [44]:
startup_deal_count = (
    df.groupby("startup_clean")
      .size()
      .rename("startup_deal_count")
)

df = df.merge(
    startup_deal_count,
    on="startup_clean",
    how="left"
)

In [45]:
df["funding_frequency"] = np.where(
    df["startup_deal_count"] > 1,
    "Repeat Funded",
    "Single Funding Record"
)

In [46]:
startup_total_funding = (
    df.groupby("startup_clean")["amount"]
      .sum()
      .rename("startup_total_funding")
)

df = df.merge(
    startup_total_funding,
    on="startup_clean",
    how="left"
)

In [47]:
sector_deal_count = (
    df.groupby("sector_group")
      .size()
      .rename("sector_deal_count")
)

df = df.merge(
    sector_deal_count,
    on="sector_group",
    how="left"
)

In [48]:
sector_total_funding = (
    df.groupby("sector_group")["amount"]
      .sum()
      .rename("sector_total_funding")
)

df = df.merge(
    sector_total_funding,
    on="sector_group",
    how="left"
)

In [49]:
df["year"] = pd.to_numeric(df["year"], errors="coerce")

In [50]:
df["year"].value_counts().sort_index()

year
2015    908
2016    968
2017    674
2018    814
2019    188
2020    997
2021    525
Name: count, dtype: int64

In [52]:
def startup_funding_profile(row):
    if row["startup_deal_count"] >= 4:
        return "Highly Repeatedly Funded"
    elif row["startup_deal_count"] >= 2:
        return "Repeat Funded"
    else:
        return "Single Funding Record"

df["startup_funding_profile"] = df.apply(
    startup_funding_profile,
    axis=1
)

In [53]:
df.columns.tolist()

['startup',
 'vertical',
 'city',
 'investors',
 'round',
 'amount',
 'year',
 'state',
 'disclosed_amount',
 'funding_disclosed',
 'funding_size',
 'sector_group',
 'round_group',
 'round_order',
 'startup_clean',
 'city_clean',
 'state_clean',
 'startup_hub',
 'funding_size_order',
 'is_disclosed',
 'mega_deal',
 'startup_deal_count',
 'funding_frequency',
 'startup_total_funding',
 'sector_deal_count',
 'sector_total_funding',
 'startup_funding_profile']

In [54]:
print("FINAL DATASET AUDIT")
print("-" * 40)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nUnique Startups:")
print(df["startup_clean"].nunique())

print("\nUnique Sectors:")
print(df["sector_group"].nunique())

print("\nStartup Hubs:")
print(df["startup_hub"].nunique())

print("\nFunding Round Groups:")
print(df["round_group"].nunique())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nFunding Disclosure:")
print(df["funding_disclosed"].value_counts())

print("\nFunding Size Distribution:")
print(df["funding_size"].value_counts())

print("\nYear Range:")
print(df["year"].min(), "-", df["year"].max())

FINAL DATASET AUDIT
----------------------------------------
Rows: 5074
Columns: 27

Unique Startups:
3139

Unique Sectors:
21

Startup Hubs:
56

Funding Round Groups:
12

Missing Values:
startup                       0
vertical                      0
city                          0
investors                     0
round                         0
amount                        0
year                          0
state                         0
disclosed_amount           1599
funding_disclosed             0
funding_size                  0
sector_group                  0
round_group                   0
round_order                   0
startup_clean                 0
city_clean                    0
state_clean                   0
startup_hub                   0
funding_size_order            0
is_disclosed                  0
mega_deal                     0
startup_deal_count            0
funding_frequency             0
startup_total_funding         0
sector_deal_count             0
sector_total

In [55]:
total_funding = df["amount"].sum()

total_deals = len(df)

disclosed_deals = df["is_disclosed"].sum()

unique_startups = df["startup_clean"].nunique()

avg_deal_size = df["disclosed_amount"].mean()

median_deal_size = df["disclosed_amount"].median()

disclosure_rate = (
    disclosed_deals / total_deals
) * 100

print(f"Total Funding: ₹{total_funding:,.2f} Cr")
print(f"Total Deals: {total_deals:,}")
print(f"Funded Startups: {unique_startups:,}")
print(f"Disclosed Deals: {disclosed_deals:,}")
print(f"Disclosure Rate: {disclosure_rate:.2f}%")
print(f"Average Disclosed Deal: ₹{avg_deal_size:,.2f} Cr")
print(f"Median Disclosed Deal: ₹{median_deal_size:,.2f} Cr")

Total Funding: ₹610,548.29 Cr
Total Deals: 5,074
Funded Startups: 3,139
Disclosed Deals: 3,475
Disclosure Rate: 68.49%
Average Disclosed Deal: ₹175.70 Cr
Median Disclosed Deal: ₹16.50 Cr


In [56]:
df.to_csv(
    "indian_startup_funding_cleaned.csv",
    index=False
)

In [57]:
powerbi_columns = [
    "startup_clean",
    "sector_group",
    "vertical",
    "city_clean",
    "startup_hub",
    "state_clean",
    "investors",
    "round",
    "round_group",
    "round_order",
    "amount",
    "disclosed_amount",
    "funding_disclosed",
    "is_disclosed",
    "funding_size",
    "funding_size_order",
    "mega_deal",
    "startup_deal_count",
    "funding_frequency",
    "startup_funding_profile",
    "year"
]

powerbi_df = df[powerbi_columns]

powerbi_df.to_csv(
    "indian_startup_funding_powerbi.csv",
    index=False
)

In [58]:
market_kpis = {
    "Total Funding (₹ Cr)": df["amount"].sum(),
    "Total Deals": len(df),
    "Disclosed Deals": df["is_disclosed"].sum(),
    "Unique Startups": df["startup_clean"].nunique(),
    "Average Deal Size (₹ Cr)": df["disclosed_amount"].mean(),
    "Median Deal Size (₹ Cr)": df["disclosed_amount"].median(),
    "Largest Deal (₹ Cr)": df["amount"].max()
}

market_kpis

{'Total Funding (₹ Cr)': np.float64(610548.29),
 'Total Deals': 5074,
 'Disclosed Deals': np.int64(3475),
 'Unique Startups': 3139,
 'Average Deal Size (₹ Cr)': np.float64(175.69734964028777),
 'Median Deal Size (₹ Cr)': np.float64(16.5),
 'Largest Deal (₹ Cr)': np.float64(16650.0)}

In [59]:
for key, value in market_kpis.items():
    if "₹" in key:
        print(f"{key}: ₹{value:,.2f}")
    else:
        print(f"{key}: {value:,.0f}")

Total Funding (₹ Cr): ₹610,548.29
Total Deals: 5,074
Disclosed Deals: 3,475
Unique Startups: 3,139
Average Deal Size (₹ Cr): ₹175.70
Median Deal Size (₹ Cr): ₹16.50
Largest Deal (₹ Cr): ₹16,650.00


In [60]:
yearly_funding = (
    df.groupby("year")
      .agg(
          total_funding=("amount", "sum"),
          total_deals=("startup_clean", "size"),
          disclosed_deals=("is_disclosed", "sum"),
          funded_startups=("startup_clean", "nunique"),
          avg_deal_size=("disclosed_amount", "mean"),
          median_deal_size=("disclosed_amount", "median")
      )
      .reset_index()
)

yearly_funding

,year,total_funding,total_deals,disclosed_deals,funded_startups,avg_deal_size,median_deal_size
0,2015,70526.54,908,641,812,110.025803,12.38
1,2016,31549.01,968,582,903,54.207921,8.25
2,2017,40665.69,674,453,623,89.769735,18.56
3,2018,43474.74,814,567,690,76.675026,6.37
4,2019,37610.76,188,101,169,372.383762,88.69
5,2020,321960.24,997,755,831,426.437404,66.00
6,2021,64761.31,525,376,524,172.237527,11.00


In [61]:
yearly_funding["funding_yoy_growth"] = (
    yearly_funding["total_funding"]
    .pct_change() * 100
)

In [62]:
yearly_funding["deal_yoy_growth"] = (
    yearly_funding["total_deals"]
    .pct_change() * 100
)

In [63]:
yearly_funding.round(2)

,year,total_funding,total_deals,disclosed_deals,funded_startups,avg_deal_size,median_deal_size,funding_yoy_growth,deal_yoy_growth
0,2015,70526.54,908,641,812,110.03,12.38,NaN,NaN
1,2016,31549.01,968,582,903,54.21,8.25,-55.27,6.61
2,2017,40665.69,674,453,623,89.77,18.56,28.90,-30.37
3,2018,43474.74,814,567,690,76.68,6.37,6.91,20.77
4,2019,37610.76,188,101,169,372.38,88.69,-13.49,-76.90
5,2020,321960.24,997,755,831,426.44,66.00,756.03,430.32
6,2021,64761.31,525,376,524,172.24,11.00,-79.89,-47.34


In [64]:
import plotly.express as px
import plotly.graph_objects as go

In [65]:
fig = px.line(
    yearly_funding,
    x="year",
    y="total_funding",
    markers=True,
    title="Indian Startup Funding Trend"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Funding (₹ Crore)",
    hovermode="x unified"
)

fig.show()

In [66]:
fig = px.line(
    yearly_funding,
    x="year",
    y="total_deals",
    markers=True,
    title="Startup Funding Deals by Year"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Number of Funding Deals",
    hovermode="x unified"
)

fig.show()

In [67]:
trend_comparison = yearly_funding[
    ["year", "total_funding", "total_deals"]
].copy()

trend_comparison["funding_index"] = (
    trend_comparison["total_funding"]
    / trend_comparison["total_funding"].iloc[0]
) * 100

trend_comparison["deal_index"] = (
    trend_comparison["total_deals"]
    / trend_comparison["total_deals"].iloc[0]
) * 100

In [68]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=trend_comparison["year"],
        y=trend_comparison["funding_index"],
        mode="lines+markers",
        name="Funding Index"
    )
)

fig.add_trace(
    go.Scatter(
        x=trend_comparison["year"],
        y=trend_comparison["deal_index"],
        mode="lines+markers",
        name="Deal Index"
    )
)

fig.add_hline(
    y=100,
    line_dash="dash",
    annotation_text="2015 Base = 100"
)

fig.update_layout(
    title="Funding Growth vs Deal Volume Growth",
    xaxis_title="Year",
    yaxis_title="Index (2015 = 100)",
    hovermode="x unified"
)

fig.show()

In [69]:
deal_size_trend = yearly_funding[
    [
        "year",
        "avg_deal_size",
        "median_deal_size"
    ]
].copy()

In [70]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=deal_size_trend["year"],
        y=deal_size_trend["avg_deal_size"],
        mode="lines+markers",
        name="Average Deal Size"
    )
)

fig.add_trace(
    go.Scatter(
        x=deal_size_trend["year"],
        y=deal_size_trend["median_deal_size"],
        mode="lines+markers",
        name="Median Deal Size"
    )
)

fig.update_layout(
    title="Average vs Median Startup Funding Deal Size",
    xaxis_title="Year",
    yaxis_title="Funding Amount (₹ Crore)",
    hovermode="x unified"
)

fig.show()

In [71]:
mega_analysis = (
    df.groupby("mega_deal")
      .agg(
          deals=("startup_clean", "size"),
          funding=("amount", "sum")
      )
      .reset_index()
)

mega_analysis

,mega_deal,deals,funding
0,Mega Deal,235,430939.61
1,Regular Deal,3240,179608.68
2,Undisclosed,1599,0.00


In [72]:
mega_funding = df.loc[
    df["mega_deal"] == "Mega Deal",
    "amount"
].sum()

total_funding = df["amount"].sum()

mega_deal_share = (
    mega_funding / total_funding
) * 100

print(
    f"Mega deals contributed "
    f"{mega_deal_share:.2f}% of total funding."
)

Mega deals contributed 70.58% of total funding.


In [73]:
mega_plot = mega_analysis[
    mega_analysis["mega_deal"] != "Undisclosed"
].copy()

fig = px.bar(
    mega_plot,
    x="mega_deal",
    y="funding",
    title="Funding Contribution by Deal Type",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Deal Type",
    yaxis_title="Total Funding (₹ Crore)"
)

fig.show()

In [74]:
mega_by_year = (
    df[df["mega_deal"] == "Mega Deal"]
    .groupby("year")
    .agg(
        mega_deals=("startup_clean", "size"),
        mega_funding=("amount", "sum")
    )
    .reset_index()
)

In [75]:
mega_by_year = mega_by_year.merge(
    yearly_funding[
        ["year", "total_funding"]
    ],
    on="year",
    how="right"
).fillna(0)

In [76]:
mega_by_year["mega_funding_share"] = (
    mega_by_year["mega_funding"]
    / mega_by_year["total_funding"]
) * 100

mega_by_year.round(2)

,year,mega_deals,mega_funding,total_funding,mega_funding_share
0,2015,23,38445.00,70526.54,54.51
1,2016,12,11913.00,31549.01,37.76
2,2017,15,18778.66,40665.69,46.18
3,2018,20,22327.15,43474.74,51.36
4,2019,18,28215.00,37610.76,75.02
5,2020,122,260036.50,321960.24,80.77
6,2021,25,51224.30,64761.31,79.10


In [77]:
fig = px.line(
    mega_by_year,
    x="year",
    y="mega_funding_share",
    markers=True,
    title="Mega Deals' Share of Total Funding by Year"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Mega Deal Funding Share (%)",
    hovermode="x unified"
)

fig.show()

In [78]:
yearly_funding["disclosure_rate"] = (
    yearly_funding["disclosed_deals"]
    / yearly_funding["total_deals"]
) * 100

In [79]:
fig = px.line(
    yearly_funding,
    x="year",
    y="disclosure_rate",
    markers=True,
    title="Funding Amount Disclosure Rate by Year"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Disclosure Rate (%)",
    yaxis_range=[0, 100],
    hovermode="x unified"
)

fig.show()

In [80]:
market_overview = yearly_funding[
    [
        "year",
        "total_funding",
        "total_deals",
        "funded_startups",
        "avg_deal_size",
        "median_deal_size",
        "funding_yoy_growth",
        "deal_yoy_growth",
        "disclosure_rate"
    ]
].round(2)

market_overview

,year,total_funding,total_deals,funded_startups,avg_deal_size,median_deal_size,funding_yoy_growth,deal_yoy_growth,disclosure_rate
0,2015,70526.54,908,812,110.03,12.38,NaN,NaN,70.59
1,2016,31549.01,968,903,54.21,8.25,-55.27,6.61,60.12
2,2017,40665.69,674,623,89.77,18.56,28.90,-30.37,67.21
3,2018,43474.74,814,690,76.68,6.37,6.91,20.77,69.66
4,2019,37610.76,188,169,372.38,88.69,-13.49,-76.90,53.72
5,2020,321960.24,997,831,426.44,66.00,756.03,430.32,75.73
6,2021,64761.31,525,524,172.24,11.00,-79.89,-47.34,71.62


In [85]:
display(
    market_overview.round({
        "total_funding": 2,
        "avg_deal_size": 2,
        "median_deal_size": 2,
        "funding_yoy_growth": 2,
        "deal_yoy_growth": 2,
        "disclosure_rate": 2
    })
)

,year,total_funding,total_deals,funded_startups,avg_deal_size,median_deal_size,funding_yoy_growth,deal_yoy_growth,disclosure_rate
0,2015,70526.54,908,812,110.03,12.38,NaN,NaN,70.59
1,2016,31549.01,968,903,54.21,8.25,-55.27,6.61,60.12
2,2017,40665.69,674,623,89.77,18.56,28.90,-30.37,67.21
3,2018,43474.74,814,690,76.68,6.37,6.91,20.77,69.66
4,2019,37610.76,188,169,372.38,88.69,-13.49,-76.90,53.72
5,2020,321960.24,997,831,426.44,66.00,756.03,430.32,75.73
6,2021,64761.31,525,524,172.24,11.00,-79.89,-47.34,71.62


In [86]:
growth_data = yearly_funding.dropna(
    subset=["funding_yoy_growth"]
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=growth_data["year"],
        y=growth_data["funding_yoy_growth"],
        name="Funding Growth"
    )
)

fig.add_trace(
    go.Bar(
        x=growth_data["year"],
        y=growth_data["deal_yoy_growth"],
        name="Deal Growth"
    )
)

fig.update_layout(
    title="Year-over-Year Funding and Deal Growth",
    xaxis_title="Year",
    yaxis_title="YoY Growth (%)",
    barmode="group"
)

fig.show()

In [87]:
sector_summary = (
    df.groupby("sector_group")
      .agg(
          total_funding=("amount", "sum"),
          total_deals=("startup_clean", "size"),
          unique_startups=("startup_clean", "nunique"),
          disclosed_deals=("is_disclosed", "sum"),
          avg_deal_size=("disclosed_amount", "mean"),
          median_deal_size=("disclosed_amount", "median")
      )
      .reset_index()
)

In [88]:
sector_summary["funding_share"] = (
    sector_summary["total_funding"]
    / sector_summary["total_funding"].sum()
) * 100

In [89]:
sector_summary["disclosure_rate"] = (
    sector_summary["disclosed_deals"]
    / sector_summary["total_deals"]
) * 100

In [90]:
sector_summary = sector_summary.sort_values(
    "total_funding",
    ascending=False
)

sector_summary.round(2)

,sector_group,total_funding,total_deals,unique_startups,disclosed_deals,avg_deal_size,median_deal_size,funding_share,disclosure_rate
8,FinTech,107771.03,598,319,471,228.81,34.65,17.65,78.76
4,E-Commerce & Retail,81428.66,524,299,359,226.82,26.00,13.34,68.51
5,EdTech,68387.70,341,208,226,302.60,15.00,11.20,66.28
9,FoodTech,49406.01,243,151,178,277.56,14.92,8.09,73.25
15,Mobility,40100.95,197,92,140,286.44,31.00,6.57,71.07
14,Media & Entertainment,34548.00,233,137,162,213.26,17.58,5.66,69.53
1,AgriTech,31774.86,93,53,69,460.51,42.00,5.20,74.19
19,Technology,30489.39,611,476,410,74.36,9.49,4.99,67.10
10,HealthTech,28798.27,408,242,256,112.49,16.00,4.72,62.75
7,Enterprise Tech,27048.84,195,110,137,197.44,22.00,4.43,70.26


In [91]:
top_sector_funding = (
    sector_summary
    .nlargest(10, "total_funding")
    .sort_values("total_funding")
)

In [92]:
fig = px.bar(
    top_sector_funding,
    x="total_funding",
    y="sector_group",
    orientation="h",
    title="Top 10 Sectors by Total Startup Funding",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Total Funding (₹ Crore)",
    yaxis_title="Sector"
)

fig.show()

In [93]:
top_sector_deals = (
    sector_summary
    .nlargest(10, "total_deals")
    .sort_values("total_deals")
)

In [94]:
fig = px.bar(
    top_sector_deals,
    x="total_deals",
    y="sector_group",
    orientation="h",
    title="Top 10 Sectors by Number of Funding Deals",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Deals",
    yaxis_title="Sector"
)

fig.show()

In [95]:
sector_avg_deal = sector_summary[
    sector_summary["disclosed_deals"] >= 10
].copy()

In [96]:
sector_avg_deal = (
    sector_avg_deal
    .nlargest(10, "avg_deal_size")
    .sort_values("avg_deal_size")
)

In [97]:
fig = px.bar(
    sector_avg_deal,
    x="avg_deal_size",
    y="sector_group",
    orientation="h",
    title="Top Sectors by Average Disclosed Deal Size",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Average Deal Size (₹ Crore)",
    yaxis_title="Sector"
)

fig.show()

In [98]:
sector_median_deal = sector_summary[
    sector_summary["disclosed_deals"] >= 10
].copy()

sector_median_deal = (
    sector_median_deal
    .nlargest(10, "median_deal_size")
    .sort_values("median_deal_size")
)

In [99]:
fig = px.bar(
    sector_median_deal,
    x="median_deal_size",
    y="sector_group",
    orientation="h",
    title="Top Sectors by Median Disclosed Deal Size",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Median Deal Size (₹ Crore)",
    yaxis_title="Sector"
)

fig.show()

In [100]:
sector_share = (
    sector_summary
    .nlargest(10, "total_funding")
    .copy()
)

sector_share[
    ["sector_group", "total_funding", "funding_share"]
].round(2)

,sector_group,total_funding,funding_share
8,FinTech,107771.03,17.65
4,E-Commerce & Retail,81428.66,13.34
5,EdTech,68387.70,11.20
9,FoodTech,49406.01,8.09
15,Mobility,40100.95,6.57
14,Media & Entertainment,34548.00,5.66
1,AgriTech,31774.86,5.20
19,Technology,30489.39,4.99
10,HealthTech,28798.27,4.72
7,Enterprise Tech,27048.84,4.43


In [101]:
fig = px.bar(
    sector_share.sort_values("funding_share"),
    x="funding_share",
    y="sector_group",
    orientation="h",
    title="Top 10 Sectors' Share of Startup Funding",
    text="funding_share"
)

fig.update_traces(
    texttemplate="%{text:.1f}%"
)

fig.update_layout(
    xaxis_title="Share of Total Funding (%)",
    yaxis_title="Sector"
)

fig.show()

In [102]:
top3_sector_share = (
    sector_summary.nlargest(3, "total_funding")["total_funding"].sum()
    / sector_summary["total_funding"].sum()
) * 100

top5_sector_share = (
    sector_summary.nlargest(5, "total_funding")["total_funding"].sum()
    / sector_summary["total_funding"].sum()
) * 100

In [103]:
print(f"Top 3 sectors account for {top3_sector_share:.2f}% of total funding.")
print(f"Top 5 sectors account for {top5_sector_share:.2f}% of total funding.")

Top 3 sectors account for 42.19% of total funding.
Top 5 sectors account for 56.85% of total funding.


In [104]:
top5_sectors = (
    sector_summary
    .nlargest(5, "total_funding")["sector_group"]
    .tolist()
)

In [105]:
sector_yearly = (
    df[df["sector_group"].isin(top5_sectors)]
    .groupby(["year", "sector_group"])["amount"]
    .sum()
    .reset_index()
)

In [106]:
fig = px.line(
    sector_yearly,
    x="year",
    y="amount",
    color="sector_group",
    markers=True,
    title="Funding Trend of Top 5 Sectors"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Funding (₹ Crore)",
    hovermode="x unified"
)

fig.show()

In [107]:
sector_deal_yearly = (
    df[df["sector_group"].isin(top5_sectors)]
    .groupby(["year", "sector_group"])
    .size()
    .reset_index(name="deals")
)

In [108]:
fig = px.line(
    sector_deal_yearly,
    x="year",
    y="deals",
    color="sector_group",
    markers=True,
    title="Deal Activity Trend of Top 5 Sectors"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Number of Deals",
    hovermode="x unified"
)

fig.show()

In [109]:
fig = px.scatter(
    sector_summary,
    x="total_deals",
    y="total_funding",
    size="unique_startups",
    hover_name="sector_group",
    title="Sector Funding vs Deal Activity",
    labels={
        "total_deals": "Number of Deals",
        "total_funding": "Total Funding (₹ Crore)",
        "unique_startups": "Unique Startups"
    }
)

fig.show()

In [110]:
sector_mega = (
    df.groupby("sector_group")
      .agg(
          total_funding=("amount", "sum"),
          mega_funding=(
              "amount",
              lambda x: x[df.loc[x.index, "mega_deal"] == "Mega Deal"].sum()
          )
      )
      .reset_index()
)

In [111]:
sector_mega["mega_deal_share"] = (
    sector_mega["mega_funding"]
    / sector_mega["total_funding"]
) * 100


In [112]:
sector_mega_analysis = (
    sector_mega[
        sector_mega["total_funding"] > 0
    ]
    .nlargest(10, "total_funding")
    .sort_values("mega_deal_share")
)

In [113]:
fig = px.bar(
    sector_mega_analysis,
    x="mega_deal_share",
    y="sector_group",
    orientation="h",
    title="Mega Deal Dependence Among Top Funded Sectors",
    text="mega_deal_share"
)

fig.update_traces(
    texttemplate="%{text:.1f}%"
)

fig.update_layout(
    xaxis_title="Funding From Mega Deals (%)",
    yaxis_title="Sector"
)

fig.show()

In [114]:
sector_summary["funding_rank"] = (
    sector_summary["total_funding"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

sector_summary["deal_rank"] = (
    sector_summary["total_deals"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

In [115]:
sector_summary["rank_gap"] = (
    sector_summary["deal_rank"]
    - sector_summary["funding_rank"]
)

In [116]:
sector_summary[
    [
        "sector_group",
        "funding_rank",
        "deal_rank",
        "rank_gap",
        "total_funding",
        "total_deals"
    ]
].sort_values("funding_rank").head(15)

,sector_group,funding_rank,deal_rank,rank_gap,total_funding,total_deals
8,FinTech,1,3,2,107771.03,598
4,E-Commerce & Retail,2,4,2,81428.66,524
5,EdTech,3,6,3,68387.70,341
9,FoodTech,4,7,3,49406.01,243
15,Mobility,5,9,4,40100.95,197
14,Media & Entertainment,6,8,2,34548.00,233
1,AgriTech,7,16,9,31774.86,93
19,Technology,8,2,-6,30489.39,611
10,HealthTech,9,5,-4,28798.27,408
7,Enterprise Tech,10,10,0,27048.84,195


In [117]:
sector_intelligence = sector_summary[
    [
        "sector_group",
        "total_funding",
        "funding_share",
        "total_deals",
        "unique_startups",
        "avg_deal_size",
        "median_deal_size",
        "funding_rank",
        "deal_rank",
        "rank_gap",
        "disclosure_rate"
    ]
].copy()

sector_intelligence = sector_intelligence.round(2)

sector_intelligence.head(15)

,sector_group,total_funding,funding_share,total_deals,unique_startups,avg_deal_size,median_deal_size,funding_rank,deal_rank,rank_gap,disclosure_rate
8,FinTech,107771.03,17.65,598,319,228.81,34.65,1,3,2,78.76
4,E-Commerce & Retail,81428.66,13.34,524,299,226.82,26.00,2,4,2,68.51
5,EdTech,68387.70,11.20,341,208,302.60,15.00,3,6,3,66.28
9,FoodTech,49406.01,8.09,243,151,277.56,14.92,4,7,3,73.25
15,Mobility,40100.95,6.57,197,92,286.44,31.00,5,9,4,71.07
14,Media & Entertainment,34548.00,5.66,233,137,213.26,17.58,6,8,2,69.53
1,AgriTech,31774.86,5.20,93,53,460.51,42.00,7,16,9,74.19
19,Technology,30489.39,4.99,611,476,74.36,9.49,8,2,-6,67.10
10,HealthTech,28798.27,4.72,408,242,112.49,16.00,9,5,-4,62.75
7,Enterprise Tech,27048.84,4.43,195,110,197.44,22.00,10,10,0,70.26


In [118]:
sector_intelligence.head(10)

,sector_group,total_funding,funding_share,total_deals,unique_startups,avg_deal_size,median_deal_size,funding_rank,deal_rank,rank_gap,disclosure_rate
8,FinTech,107771.03,17.65,598,319,228.81,34.65,1,3,2,78.76
4,E-Commerce & Retail,81428.66,13.34,524,299,226.82,26.00,2,4,2,68.51
5,EdTech,68387.70,11.20,341,208,302.60,15.00,3,6,3,66.28
9,FoodTech,49406.01,8.09,243,151,277.56,14.92,4,7,3,73.25
15,Mobility,40100.95,6.57,197,92,286.44,31.00,5,9,4,71.07
14,Media & Entertainment,34548.00,5.66,233,137,213.26,17.58,6,8,2,69.53
1,AgriTech,31774.86,5.20,93,53,460.51,42.00,7,16,9,74.19
19,Technology,30489.39,4.99,611,476,74.36,9.49,8,2,-6,67.10
10,HealthTech,28798.27,4.72,408,242,112.49,16.00,9,5,-4,62.75
7,Enterprise Tech,27048.84,4.43,195,110,197.44,22.00,10,10,0,70.26


In [119]:
print(f"Top 3 sectors: {top3_sector_share:.2f}%")
print(f"Top 5 sectors: {top5_sector_share:.2f}%")

Top 3 sectors: 42.19%
Top 5 sectors: 56.85%


In [120]:
sector_mega_analysis[
    ["sector_group", "mega_deal_share"]
]

,sector_group,mega_deal_share
19,Technology,43.785723
10,HealthTech,51.246134
8,FinTech,69.333104
14,Media & Entertainment,72.664264
4,E-Commerce & Retail,73.682595
7,Enterprise Tech,76.921968
15,Mobility,79.520261
1,AgriTech,81.998788
9,FoodTech,83.230765
5,EdTech,84.500941


In [121]:
startup_summary = (
    df.groupby("startup_clean")
      .agg(
          total_funding=("amount", "sum"),
          total_deals=("startup_clean", "size"),
          disclosed_deals=("is_disclosed", "sum"),
          avg_deal_size=("disclosed_amount", "mean"),
          median_deal_size=("disclosed_amount", "median"),
          primary_sector=("sector_group", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"),
          primary_hub=("startup_hub", lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
      )
      .reset_index()
)

In [124]:
startup_summary = startup_summary.sort_values(
    "total_funding",
    ascending=False
)
startup_summary.head(10)

,startup_clean,total_funding,total_deals,disclosed_deals,avg_deal_size,median_deal_size,primary_sector,primary_hub
365,Byju'S,31445.25,15,10,3144.525000,1075.750,EdTech,Bengaluru
1949,Ola,17681.16,18,16,1105.072500,456.250,Mobility,Bengaluru
2617,Swiggy,13513.63,12,11,1228.511818,288.750,FoodTech,Bengaluru
927,Flipkart,12560.28,7,7,1794.325714,412.500,E-Commerce & Retail,Bengaluru
2249,Reliance Retail Ventures,12315.00,4,4,3078.750000,2704.500,Services,Mumbai
3122,Zomato,10590.75,9,9,1176.750000,514.000,FoodTech,Delhi NCR
2825,Udaan,9450.00,6,6,1575.000000,1134.375,Enterprise Tech,Bengaluru
1900,Ninjacart,9092.63,8,7,1298.947143,214.500,AgriTech,Bengaluru
1533,Krimanshi Feeds,8491.00,4,3,2830.333333,166.000,AgriTech,Jodhpur
2896,Verse Innovation,8300.00,1,1,8300.000000,8300.000,Media & Entertainment,Bengaluru


In [125]:
top_startups = (
    startup_summary
    .nlargest(15, "total_funding")
    .sort_values("total_funding")
)

fig = px.bar(
    top_startups,
    x="total_funding",
    y="startup_clean",
    orientation="h",
    title="Top 15 Startups by Total Funding",
    text_auto=".2s",
    hover_data=[
        "primary_sector",
        "total_deals",
        "primary_hub"
    ]
)

fig.update_layout(
    xaxis_title="Total Funding (₹ Crore)",
    yaxis_title="Startup"
)

fig.show()

In [126]:
total_market_funding = startup_summary["total_funding"].sum()

top5_startup_share = (
    startup_summary.nlargest(5, "total_funding")["total_funding"].sum()
    / total_market_funding
) * 100

top10_startup_share = (
    startup_summary.nlargest(10, "total_funding")["total_funding"].sum()
    / total_market_funding
) * 100

top20_startup_share = (
    startup_summary.nlargest(20, "total_funding")["total_funding"].sum()
    / total_market_funding
) * 100

print(f"Top 5 startups: {top5_startup_share:.2f}%")
print(f"Top 10 startups: {top10_startup_share:.2f}%")
print(f"Top 20 startups: {top20_startup_share:.2f}%")

Top 5 startups: 14.33%
Top 10 startups: 21.86%
Top 20 startups: 33.39%


In [128]:
frequent_startups = (
    startup_summary
    .sort_values(
        ["total_deals", "total_funding"],
        ascending=[False, False]
    )
    .head(15)
    .sort_values("total_deals")
)

In [129]:
fig = px.bar(
    frequent_startups,
    x="total_deals",
    y="startup_clean",
    orientation="h",
    title="Most Frequently Funded Startups",
    text_auto=True,
    hover_data=[
        "total_funding",
        "primary_sector"
    ]
)

fig.update_layout(
    xaxis_title="Number of Funding Records",
    yaxis_title="Startup"
)

fig.show()

In [130]:
startup_scatter = startup_summary[
    startup_summary["total_funding"] > 0
].copy()

In [131]:
fig = px.scatter(
    startup_scatter,
    x="total_deals",
    y="total_funding",
    hover_name="startup_clean",
    color="primary_sector",
    title="Startup Funding vs Funding Frequency",
    labels={
        "total_deals": "Number of Funding Records",
        "total_funding": "Total Funding (₹ Crore)",
        "primary_sector": "Sector"
    }
)

fig.show()

In [132]:
largest_rounds = (
    df[df["amount"] > 0]
    .nlargest(15, "amount")
    [
        [
            "startup_clean",
            "sector_group",
            "round_group",
            "amount",
            "year",
            "investors"
        ]
    ]
)

largest_rounds

,startup_clean,sector_group,round_group,amount,year,investors
3673,Byju'S,EdTech,Undisclosed / Other,16650.0,2020,"Owl Ventures, Tiger Global Management"
4990,Swiggy,FoodTech,Late Stage,10000.0,2021,Undisclosed
3889,Galaxycard,FinTech,Undisclosed / Other,8300.0,2020,"Alibaba Group, InnoVen Capital"
4027,Krimanshi Feeds,AgriTech,Undisclosed / Other,8300.0,2020,"Canaan Partners, NewView Capital"
4115,Mooofarm,AgriTech,Undisclosed / Other,8300.0,2020,"Poonawalla, Allana Group"
4138,Ninjacart,AgriTech,Undisclosed / Other,8300.0,2020,"Tiger Global Management, Sequoia Capital India"
4476,Verse Innovation,Media & Entertainment,Undisclosed / Other,8300.0,2020,Google
4947,Sbi Life,FinTech,Private Equity,7420.0,2021,Undisclosed
3648,Blacksoil,FinTech,Undisclosed / Other,6640.0,2020,Small Industries Development Bank of India
3737,Cred,FinTech,Undisclosed / Other,6640.0,2020,"General Catalyst, Ribbit Capital"


In [133]:
largest_rounds_plot = largest_rounds.sort_values("amount")

fig = px.bar(
    largest_rounds_plot,
    x="amount",
    y="startup_clean",
    orientation="h",
    title="Largest Individual Startup Funding Rounds",
    text_auto=".2s",
    hover_data=[
        "year",
        "sector_group",
        "round_group",
        "investors"
    ]
)

fig.update_layout(
    xaxis_title="Funding Amount (₹ Crore)",
    yaxis_title="Startup"
)

fig.show()

In [134]:
round_summary = (
    df.groupby(
        ["round_group", "round_order"]
    )
    .agg(
        total_funding=("amount", "sum"),
        total_deals=("startup_clean", "size"),
        unique_startups=("startup_clean", "nunique"),
        disclosed_deals=("is_disclosed", "sum"),
        avg_deal_size=("disclosed_amount", "mean"),
        median_deal_size=("disclosed_amount", "median")
    )
    .reset_index()
    .sort_values("round_order")
)

round_summary.round(2)

,round_group,round_order,total_funding,total_deals,unique_startups,disclosed_deals,avg_deal_size,median_deal_size
9,Seed / Angel,1,10836.00,2129,1587,1222,8.87,2.89
3,Early Stage,2,4762.07,177,100,148,32.18,15.00
5,Growth Stage,3,26450.91,104,61,87,304.03,124.65
6,Late Stage,4,30263.65,36,24,33,917.08,371.25
11,Venture - Unspecified,5,7137.21,79,42,55,129.77,40.00
7,Private Equity,6,195464.98,1357,971,1082,180.65,49.50
2,Debt,7,5685.42,59,39,54,105.29,34.65
0,Corporate,8,3163.06,18,10,10,316.31,26.00
8,Public / Secondary,9,6035.50,12,6,9,670.61,140.00
4,Grant / Non-Equity,10,41.42,11,6,7,5.92,0.40


In [135]:
round_summary = (
    df.groupby(
        ["round_group", "round_order"]
    )
    .agg(
        total_funding=("amount", "sum"),
        total_deals=("startup_clean", "size"),
        unique_startups=("startup_clean", "nunique"),
        disclosed_deals=("is_disclosed", "sum"),
        avg_deal_size=("disclosed_amount", "mean"),
        median_deal_size=("disclosed_amount", "median")
    )
    .reset_index()
    .sort_values("round_order")
)

round_summary.round(2)

,round_group,round_order,total_funding,total_deals,unique_startups,disclosed_deals,avg_deal_size,median_deal_size
9,Seed / Angel,1,10836.00,2129,1587,1222,8.87,2.89
3,Early Stage,2,4762.07,177,100,148,32.18,15.00
5,Growth Stage,3,26450.91,104,61,87,304.03,124.65
6,Late Stage,4,30263.65,36,24,33,917.08,371.25
11,Venture - Unspecified,5,7137.21,79,42,55,129.77,40.00
7,Private Equity,6,195464.98,1357,971,1082,180.65,49.50
2,Debt,7,5685.42,59,39,54,105.29,34.65
0,Corporate,8,3163.06,18,10,10,316.31,26.00
8,Public / Secondary,9,6035.50,12,6,9,670.61,140.00
4,Grant / Non-Equity,10,41.42,11,6,7,5.92,0.40


In [136]:
fig = px.bar(
    round_summary,
    x="round_group",
    y="total_deals",
    title="Funding Deal Activity by Stage",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Funding Stage",
    yaxis_title="Number of Deals",
    xaxis_tickangle=-35
)

fig.show()

In [137]:
round_avg = round_summary[
    round_summary["disclosed_deals"] >= 10
].copy()

In [139]:
fig = px.bar(
    round_avg,
    x="round_group",
    y="avg_deal_size",
    title="Average Disclosed Deal Size by Funding Stage",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Funding Stage",
    yaxis_title="Average Deal Size (₹ Crore)",
    xaxis_tickangle=-35
)

fig.show()

In [140]:
round_yearly = (
    df.groupby(["year", "round_group"])["amount"]
      .sum()
      .reset_index()
)

In [141]:
major_rounds = [
    "Seed / Angel",
    "Early Stage",
    "Growth Stage",
    "Late Stage",
    "Private Equity",
    "Venture - Unspecified"
]

round_yearly_major = round_yearly[
    round_yearly["round_group"].isin(major_rounds)
]

In [142]:
fig = px.line(
    round_yearly_major,
    x="year",
    y="amount",
    color="round_group",
    markers=True,
    title="Funding Stage Trends Over Time"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Funding (₹ Crore)",
    hovermode="x unified"
)

fig.show()

In [143]:
df["investors"].value_counts()

investors
Undisclosed                         1105
Undisclosed investors                 80
Undisclosed investor                  24
Indian angel network                  23
Ratan tata                            19
                                    ... 
Pacific Horizon                        1
Sony Innovation Fund                   1
Godrej Investments                     1
Gemba Capital, Gunderson Dettmer       1
Hindustan Media Ventures               1
Name: count, Length: 3190, dtype: Int64

In [144]:
investor_df = df[
    [
        "startup_clean",
        "sector_group",
        "amount",
        "year",
        "investors"
    ]
].copy()

In [145]:
investor_df["investors"] = (
    investor_df["investors"]
    .fillna("Undisclosed")
    .str.split(",")
)

In [146]:
investor_df = investor_df.explode("investors")

In [147]:
investor_df["investor"] = (
    investor_df["investors"]
    .str.strip()
)

In [148]:
investor_df = investor_df.drop(columns="investors")

In [149]:
investor_df.head()

,startup_clean,sector_group,amount,year,investor
0,#Fame,Media & Entertainment,82.5,2015,To the new ventures
1,1Mg,HealthTech,49.5,2015,Deep kalra
1,1Mg,HealthTech,49.5,2015,Sequoia capital
1,1Mg,HealthTech,49.5,2015,Omidyar network
1,1Mg,HealthTech,49.5,2015,Intel capital & kae capital


In [150]:
investor_df["investor"].value_counts().head(20)

investor
Undisclosed               1105
Undisclosed investors       82
Sequoia capital             72
Accel partners              69
                            61
Saif partners               50
Kalaari capital             49
Blume ventures              47
Indian angel network        42
Nexus venture partners      31
Idg ventures                29
Venture Catalysts           29
Rajan anandan               27
Tiger global                27
Ratan tata                  25
Kae capital                 25
Matrix partners             25
Undisclosed investor        24
Beenext                     23
India quotient              22
Name: count, dtype: int64

In [151]:
investor_df["investor"] = investor_df["investor"].replace({
    "Undisclosed Investors": "Undisclosed",
    "Undisclosed investors": "Undisclosed",
    "undisclosed": "Undisclosed"
})

In [152]:
known_investors = investor_df[
    investor_df["investor"] != "Undisclosed"
].copy()

In [153]:
investor_activity = (
    known_investors
    .groupby("investor")
    .agg(
        investment_records=("startup_clean", "size"),
        startups_backed=("startup_clean", "nunique")
    )
    .reset_index()
)

In [154]:
top_investors = (
    investor_activity
    .nlargest(20, "investment_records")
    .sort_values("investment_records")
)

In [155]:
fig = px.bar(
    top_investors,
    x="investment_records",
    y="investor",
    orientation="h",
    title="Most Active Startup Investors",
    text_auto=True,
    hover_data=["startups_backed"]
)

fig.update_layout(
    xaxis_title="Investment Records",
    yaxis_title="Investor"
)

fig.show()

In [156]:
top_portfolio_investors = (
    investor_activity
    .nlargest(20, "startups_backed")
    .sort_values("startups_backed")
)

In [157]:
fig = px.bar(
    top_portfolio_investors,
    x="startups_backed",
    y="investor",
    orientation="h",
    title="Top Investors by Number of Startups Backed",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Unique Startups Backed",
    yaxis_title="Investor"
)

fig.show()

In [158]:
top10_investor_names = (
    investor_activity
    .nlargest(10, "investment_records")["investor"]
    .tolist()
)

In [159]:
investor_sector = (
    known_investors[
        known_investors["investor"].isin(top10_investor_names)
    ]
    .groupby(["investor", "sector_group"])
    .size()
    .reset_index(name="investments")
)

In [161]:
investor_top_sector = (
    investor_sector
    .sort_values(
        ["investor", "investments"],
        ascending=[True, False]
    )
    .drop_duplicates("investor")
)

investor_top_sector

,investor,sector_group,investments
2,,Consumer Internet,21
27,Accel partners,Technology,11
31,Blume ventures,Consumer Internet,15
55,Idg ventures,Technology,6
59,Indian angel network,Consumer Internet,14
81,Kalaari capital,Technology,13
84,Nexus venture partners,Consumer Internet,6
95,Saif partners,Consumer Internet,10
108,Sequoia capital,E-Commerce & Retail,15
127,Venture Catalysts,FinTech,7


In [162]:
known_investors.groupby("investor")["amount"].sum()

investor
                            3673.77
"kedaraa capital             160.88
& other angel investors.       0.00
& others                       8.25
021 Capital                  428.00
                             ...   
varaprasad reddy              41.25
vikas malpani                  0.00
vikram lal                     0.00
viswanath ramachandran         2.89
vivek bhargava                 1.65
Name: amount, Length: 4278, dtype: float64

In [163]:
startup_intelligence = startup_summary[
    [
        "startup_clean",
        "primary_sector",
        "primary_hub",
        "total_funding",
        "total_deals",
        "disclosed_deals",
        "avg_deal_size",
        "median_deal_size"
    ]
].copy()

startup_intelligence["funding_rank"] = (
    startup_intelligence["total_funding"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

startup_intelligence = startup_intelligence.sort_values(
    "funding_rank"
)

startup_intelligence.head(20)

,startup_clean,primary_sector,primary_hub,total_funding,total_deals,disclosed_deals,avg_deal_size,median_deal_size,funding_rank
365,Byju'S,EdTech,Bengaluru,31445.25,15,10,3144.525000,1075.750,1
1949,Ola,Mobility,Bengaluru,17681.16,18,16,1105.072500,456.250,2
2617,Swiggy,FoodTech,Bengaluru,13513.63,12,11,1228.511818,288.750,3
927,Flipkart,E-Commerce & Retail,Bengaluru,12560.28,7,7,1794.325714,412.500,4
2249,Reliance Retail Ventures,Services,Mumbai,12315.00,4,4,3078.750000,2704.500,5
3122,Zomato,FoodTech,Delhi NCR,10590.75,9,9,1176.750000,514.000,6
2825,Udaan,Enterprise Tech,Bengaluru,9450.00,6,6,1575.000000,1134.375,7
1900,Ninjacart,AgriTech,Bengaluru,9092.63,8,7,1298.947143,214.500,8
1533,Krimanshi Feeds,AgriTech,Jodhpur,8491.00,4,3,2830.333333,166.000,9
2896,Verse Innovation,Media & Entertainment,Bengaluru,8300.00,1,1,8300.000000,8300.000,10


In [164]:
hub_summary = (
    df.groupby("startup_hub")
      .agg(
          total_funding=("amount", "sum"),
          total_deals=("startup_clean", "size"),
          unique_startups=("startup_clean", "nunique"),
          disclosed_deals=("is_disclosed", "sum"),
          avg_deal_size=("disclosed_amount", "mean"),
          median_deal_size=("disclosed_amount", "median")
      )
      .reset_index()
)

In [165]:
hub_summary = (
    df.groupby("startup_hub")
      .agg(
          total_funding=("amount", "sum"),
          total_deals=("startup_clean", "size"),
          unique_startups=("startup_clean", "nunique"),
          disclosed_deals=("is_disclosed", "sum"),
          avg_deal_size=("disclosed_amount", "mean"),
          median_deal_size=("disclosed_amount", "median")
      )
      .reset_index()
)

In [166]:
hub_summary["disclosure_rate"] = (
    hub_summary["disclosed_deals"]
    / hub_summary["total_deals"]
) * 100

In [168]:
hub_summary = hub_summary.sort_values(
    "total_funding",
    ascending=False
)

hub_summary.round(2)

,startup_hub,total_funding,total_deals,unique_startups,disclosed_deals,avg_deal_size,median_deal_size,disclosure_rate
7,Bengaluru,253840.80,1643,947,1166,217.70,23.55,70.97
13,Delhi NCR,145067.78,1502,938,990,146.53,12.38,65.91
36,Mumbai,122315.60,971,631,681,179.61,18.15,70.13
11,Chennai,20684.92,203,124,158,130.92,24.88,77.83
42,Pune,13240.93,194,134,124,106.78,11.69,63.92
20,Hyderabad,10072.53,162,128,122,82.56,9.49,75.31
23,Jodhpur,8493.32,7,3,6,1415.55,13.16,85.71
48,Thane,7921.68,10,5,7,1131.67,10.00,70.00
22,Jaipur,6952.70,63,45,38,182.97,19.50,60.32
30,Kolkata,6164.34,46,36,28,220.16,13.50,60.87


In [169]:
top_hubs_funding = (
    hub_summary
    .nlargest(10, "total_funding")
    .sort_values("total_funding")
)

In [170]:
fig = px.bar(
    top_hubs_funding,
    x="total_funding",
    y="startup_hub",
    orientation="h",
    title="Top Startup Hubs by Total Funding",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Total Funding (₹ Crore)",
    yaxis_title="Startup Hub"
)

fig.show()

In [172]:
top_hubs_deals = (
    hub_summary
    .nlargest(10, "total_deals")
    .sort_values("total_deals")
)

In [173]:
fig = px.bar(
    top_hubs_deals,
    x="total_deals",
    y="startup_hub",
    orientation="h",
    title="Top Startup Hubs by Funding Deal Activity",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Funding Deals",
    yaxis_title="Startup Hub"
)

fig.show()

In [174]:
top3_hub_share = (
    hub_summary.nlargest(3, "total_funding")["total_funding"].sum()
    / hub_summary["total_funding"].sum()
) * 100

top5_hub_share = (
    hub_summary.nlargest(5, "total_funding")["total_funding"].sum()
    / hub_summary["total_funding"].sum()
) * 100

In [175]:
print(f"Top 3 startup hubs account for {top3_hub_share:.2f}% of total funding.")
print(f"Top 5 startup hubs account for {top5_hub_share:.2f}% of total funding.")

Top 3 startup hubs account for 85.37% of total funding.
Top 5 startup hubs account for 90.93% of total funding.


In [176]:
fig = px.scatter(
    hub_summary[
        hub_summary["total_funding"] > 0
    ],
    x="total_deals",
    y="total_funding",
    size="unique_startups",
    hover_name="startup_hub",
    title="Startup Hub Funding vs Deal Activity",
    labels={
        "total_deals": "Number of Deals",
        "total_funding": "Total Funding (₹ Crore)",
        "unique_startups": "Unique Startups"
    }
)

fig.show()

In [177]:
hub_avg_deal = (
    hub_summary[
        hub_summary["disclosed_deals"] >= 10
    ]
    .nlargest(10, "avg_deal_size")
    .sort_values("avg_deal_size")
)

In [178]:
fig = px.bar(
    hub_avg_deal,
    x="avg_deal_size",
    y="startup_hub",
    orientation="h",
    title="Average Disclosed Deal Size by Startup Hub",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Average Deal Size (₹ Crore)",
    yaxis_title="Startup Hub"
)

fig.show()

In [180]:
hub_median_deal = (
    hub_summary[
        hub_summary["disclosed_deals"] >= 10
    ]
    .nlargest(10, "median_deal_size")
    .sort_values("median_deal_size")
)

In [181]:
fig = px.bar(
    hub_median_deal,
    x="median_deal_size",
    y="startup_hub",
    orientation="h",
    title="Median Disclosed Deal Size by Startup Hub",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Median Deal Size (₹ Crore)",
    yaxis_title="Startup Hub"
)

fig.show()

In [182]:
top5_hubs = (
    hub_summary
    .nlargest(5, "total_funding")["startup_hub"]
    .tolist()
)

In [183]:
hub_yearly = (
    df[
        df["startup_hub"].isin(top5_hubs)
    ]
    .groupby(["year", "startup_hub"])["amount"]
    .sum()
    .reset_index()
)

In [184]:
fig = px.line(
    hub_yearly,
    x="year",
    y="amount",
    color="startup_hub",
    markers=True,
    title="Funding Trend of Top Startup Hubs"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Funding (₹ Crore)",
    hovermode="x unified"
)

fig.show()

In [185]:
hub_deals_yearly = (
    df[
        df["startup_hub"].isin(top5_hubs)
    ]
    .groupby(["year", "startup_hub"])
    .size()
    .reset_index(name="deals")
)

In [186]:
fig = px.line(
    hub_deals_yearly,
    x="year",
    y="deals",
    color="startup_hub",
    markers=True,
    title="Deal Activity Trend of Top Startup Hubs"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Number of Deals",
    hovermode="x unified"
)

fig.show()

In [187]:
hub_sector = (
    df.groupby(
        ["startup_hub", "sector_group"]
    )
    .agg(
        total_funding=("amount", "sum"),
        total_deals=("startup_clean", "size")
    )
    .reset_index()
)

In [188]:
hub_sector_top = hub_sector[
    hub_sector["startup_hub"].isin(top5_hubs)
].copy()

In [189]:
dominant_hub_sector = (
    hub_sector_top
    .sort_values(
        ["startup_hub", "total_funding"],
        ascending=[True, False]
    )
    .drop_duplicates("startup_hub")
)

dominant_hub_sector[
    [
        "startup_hub",
        "sector_group",
        "total_funding",
        "total_deals"
    ]
]

,startup_hub,sector_group,total_funding,total_deals
26,Bengaluru,EdTech,44839.93,115
66,Chennai,FinTech,6925.48,45
86,Delhi NCR,E-Commerce & Retail,26248.57,182
202,Mumbai,FinTech,34514.04,140
232,Pune,E-Commerce & Retail,2790.63,16


In [190]:
top8_sectors = (
    sector_summary
    .nlargest(8, "total_funding")["sector_group"]
    .tolist()
)

In [191]:
heatmap_data = hub_sector[
    hub_sector["startup_hub"].isin(top5_hubs)
    & hub_sector["sector_group"].isin(top8_sectors)
]

In [192]:
hub_sector_pivot = heatmap_data.pivot_table(
    index="startup_hub",
    columns="sector_group",
    values="total_funding",
    aggfunc="sum",
    fill_value=0
)

In [193]:
fig = px.imshow(
    hub_sector_pivot,
    aspect="auto",
    title="Startup Hub × Sector Funding Heatmap",
    labels={
        "x": "Sector",
        "y": "Startup Hub",
        "color": "Funding (₹ Cr)"
    }
)

fig.show()

In [194]:
hub_sector["hub_total_funding"] = (
    hub_sector
    .groupby("startup_hub")["total_funding"]
    .transform("sum")
)

In [195]:
hub_sector["sector_share_within_hub"] = (
    hub_sector["total_funding"]
    / hub_sector["hub_total_funding"]
) * 100

In [196]:
hub_specialization = (
    hub_sector[
        hub_sector["startup_hub"].isin(top5_hubs)
    ]
    .sort_values(
        ["startup_hub", "sector_share_within_hub"],
        ascending=[True, False]
    )
    .groupby("startup_hub")
    .head(3)
)

In [198]:
hub_specialization[
    [
        "startup_hub",
        "sector_group",
        "sector_share_within_hub"
    ]
].round(2)

,startup_hub,sector_group,sector_share_within_hub
26,Bengaluru,EdTech,17.66
25,Bengaluru,E-Commerce & Retail,15.95
29,Bengaluru,FinTech,13.49
66,Chennai,FinTech,33.48
64,Chennai,EdTech,21.02
75,Chennai,Technology,8.13
86,Delhi NCR,E-Commerce & Retail,18.09
90,Delhi NCR,FinTech,17.33
91,Delhi NCR,FoodTech,17.25
202,Mumbai,FinTech,28.22


In [199]:
state_summary = (
    df.groupby("state_clean")
      .agg(
          total_funding=("amount", "sum"),
          total_deals=("startup_clean", "size"),
          unique_startups=("startup_clean", "nunique")
      )
      .reset_index()
      .sort_values(
          "total_funding",
          ascending=False
      )
)

In [200]:
top_states = (
    state_summary
    .nlargest(10, "total_funding")
    .sort_values("total_funding")
)

fig = px.bar(
    top_states,
    x="total_funding",
    y="state_clean",
    orientation="h",
    title="Top States by Startup Funding",
    text_auto=".2s"
)

fig.update_layout(
    xaxis_title="Total Funding (₹ Crore)",
    yaxis_title="State"
)

fig.show()

In [201]:
hub_summary["funding_rank"] = (
    hub_summary["total_funding"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

hub_summary["deal_rank"] = (
    hub_summary["total_deals"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

hub_summary["rank_gap"] = (
    hub_summary["deal_rank"]
    - hub_summary["funding_rank"]
)

In [202]:
hub_summary[
    [
        "startup_hub",
        "funding_rank",
        "deal_rank",
        "rank_gap",
        "total_funding",
        "total_deals"
    ]
].head(15)

,startup_hub,funding_rank,deal_rank,rank_gap,total_funding,total_deals
7,Bengaluru,1,1,0,253840.80,1643
13,Delhi NCR,2,2,0,145067.78,1502
36,Mumbai,3,3,0,122315.60,971
11,Chennai,4,4,0,20684.92,203
42,Pune,5,5,0,13240.93,194
20,Hyderabad,6,6,0,10072.53,162
23,Jodhpur,7,18,11,8493.32,7
48,Thane,8,16,8,7921.68,10
22,Jaipur,9,8,-1,6952.70,63
30,Kolkata,10,9,-1,6164.34,46


In [204]:
hub_summary["funding_share"] = (
    hub_summary["total_funding"]
    / hub_summary["total_funding"].sum()
) * 100

In [205]:
hub_summary["disclosure_rate"] = (
    hub_summary["disclosed_deals"]
    / hub_summary["total_deals"]
) * 100

hub_summary["funding_rank"] = (
    hub_summary["total_funding"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

hub_summary["deal_rank"] = (
    hub_summary["total_deals"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

hub_summary["rank_gap"] = (
    hub_summary["deal_rank"]
    - hub_summary["funding_rank"]
)

In [207]:
print(hub_summary.columns.tolist())

['startup_hub', 'total_funding', 'total_deals', 'unique_startups', 'disclosed_deals', 'avg_deal_size', 'median_deal_size', 'disclosure_rate', 'funding_rank', 'deal_rank', 'rank_gap', 'funding_share']


In [208]:
geographic_intelligence = hub_summary[
    [
        "startup_hub",
        "total_funding",
        "funding_share",
        "total_deals",
        "unique_startups",
        "avg_deal_size",
        "median_deal_size",
        "funding_rank",
        "deal_rank",
        "rank_gap",
        "disclosure_rate"
    ]
].copy()

geographic_intelligence = (
    geographic_intelligence
    .sort_values("funding_rank")
    .round(2)
)

geographic_intelligence.head(15)

,startup_hub,total_funding,funding_share,total_deals,unique_startups,avg_deal_size,median_deal_size,funding_rank,deal_rank,rank_gap,disclosure_rate
7,Bengaluru,253840.80,41.58,1643,947,217.70,23.55,1,1,0,70.97
13,Delhi NCR,145067.78,23.76,1502,938,146.53,12.38,2,2,0,65.91
36,Mumbai,122315.60,20.03,971,631,179.61,18.15,3,3,0,70.13
11,Chennai,20684.92,3.39,203,124,130.92,24.88,4,4,0,77.83
42,Pune,13240.93,2.17,194,134,106.78,11.69,5,5,0,63.92
20,Hyderabad,10072.53,1.65,162,128,82.56,9.49,6,6,0,75.31
23,Jodhpur,8493.32,1.39,7,3,1415.55,13.16,7,18,11,85.71
48,Thane,7921.68,1.30,10,5,1131.67,10.00,8,16,8,70.00
22,Jaipur,6952.70,1.14,63,45,182.97,19.50,9,8,-1,60.32
30,Kolkata,6164.34,1.01,46,36,220.16,13.50,10,9,-1,60.87
